# LanceDB vector database

In [ ]:
import lancedb

# in python script: Path(__file__).parent / "vector_database"
db = lancedb.connect(uri="vector_database")
db

LanceDBConnection(uri='/Users/aigineer/Documents/github/ai_engineering_kokchun_giang/code-alongs/13_lancedb/vector_database')

In [2]:
db.uri

'/Users/aigineer/Documents/github/ai_engineering_kokchun_giang/code-alongs/13_lancedb/vector_database'

## Read in data

In [3]:
import json 
with open("data/animals_text_embeddings.json", "r") as file:
    data = json.loads(file.read())

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

## Create table

In [4]:
db.create_table("animals", exist_ok=True, data=data)


LanceTable(name='animals', version=1, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/github/ai_engineering_kokchun_giang/code-alongs/13_lancedb/vector_database'))

In [9]:
db.list_tables()

ListTablesResponse(tables=['animals'], page_token=None)

In [11]:
db["animals"]

LanceTable(name='animals', version=1, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/github/ai_engineering_kokchun_giang/code-alongs/13_lancedb/vector_database'))

In [13]:
db["animals"].head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

convert to a classic dataframe

In [15]:
df_animals = db["animals"].to_pandas()
df_animals

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"


In [18]:
df_animals.iloc[2]["text"], df_animals.iloc[2]["vector"]

('A large gray elephant drinking water.',
 array([0.88, 0.22, 0.55], dtype=float32))

to add more data

In [21]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

db["animals"].add(more_data)

AddResult(version=2)

In [22]:
db["animals"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Create empty table
- create an empty table first and then place in data 
- need to provide a schema

In [26]:
from lancedb.pydantic import LanceModel

class EmployeeSchema(LanceModel):
    first_name: str 
    last_name: str 
    salary: int

db.create_table(name = "employees", schema=EmployeeSchema, exist_ok=True)

LanceTable(name='employees', version=1, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/github/ai_engineering_kokchun_giang/code-alongs/13_lancedb/vector_database'))

In [ ]:
data = [{"first_name": "Bibbi", "last_name": "Babblarna", "salary": 1000}]
db["employees"].add(data)

AddResult(version=2)

In [30]:
db["employees"].to_pandas()

,first_name,last_name,salary
0,Bibbi,Babblarna,1000


In [32]:
db.list_tables()

ListTablesResponse(tables=['animals', 'employees'], page_token=None)

In [33]:
db.drop_table("employees")

In [34]:
db.list_tables()

ListTablesResponse(tables=['animals'], page_token=None)

## Vector search

ANN - approximate nearest neighbour for vector search

1. send in a query vector directly and search
    - this requires that we embed our query first using same embedding as what was used in the knowledge base
2. send in a text and let lancedb automatically embed it and search


In [36]:
db["animals"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [43]:
# assume that we embed our question using same embedding model as the one for animals
# question about elephant
query_vector = [0.9, 0.2, 0.5]

db["animals"].search(query_vector).limit(4).to_pandas()

,text,vector,_distance
0,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]",0.0033
1,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]",0.0094
2,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.0573
3,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.2834


## Embeddings API

- let lancedb embed our documents automatically
- let lancedb embed our query automatically and search using natural language

